## Vérification (copie, suppréssion, déplacement de fichiers)

In [111]:
import os
import numpy as np
import shutil
from tqdm import tqdm

# rename

root = '../data/PET-EARL/domain_a100/'

for subject in tqdm(os.listdir(root), position=0, desc='Renaming files', leave=True):
    subject_folder = os.path.join(root, subject)
    if os.path.isdir(subject_folder):
        for file in os.listdir(subject_folder):
            for dsr in range(1, 31):
                m, n = f'predicted_EARL_unet_DSR{dsr}_EARL2', f'pseudo-earl2-dsr{dsr}'
                if m in file:
                    old_file_path = os.path.join(subject_folder, file)
                    new_file_name = file.replace(m, n)
                    new_file_path = os.path.join(subject_folder, new_file_name)
                    shutil.move(old_file_path, new_file_path)

Renaming files:   0%|          | 0/148 [00:00<?, ?it/s]

Renaming files: 100%|██████████| 148/148 [00:00<00:00, 1955.21it/s]


In [ ]:
import os
import numpy as np
import shutil
from tqdm import tqdm

# delete files

root = 'outputs/pseudo-earl/'

for subject in tqdm(os.listdir(root), position=0, desc='Deleting files', leave=True):
    subject_folder = os.path.join(root, subject)
    if os.path.isdir(subject_folder):
        for file in os.listdir(subject_folder):
            if file.endswith('.csv'):
                file_path = os.path.join(subject_folder, file)
                os.remove(file_path)

Deleting files: 100%|██████████| 148/148 [00:00<00:00, 1495.60it/s]


In [181]:
import os
import glob
import pandas as pd
from tqdm import tqdm

input_dir = '../outputs/pseudo-earl/'

search_pattern = os.path.join(input_dir, "**", "*radiomics.csv")
csv_files = glob.glob(search_pattern, recursive=True)

assert csv_files, "Aucun fichier CSV de radiomiques trouvé dans le répertoire spécifié."
    
print(f"📄 {len(csv_files)} fichiers trouvés. Début de la lecture...")

df_list = []
for file_path in tqdm(csv_files, desc="Lecture ..."):
    df = pd.read_csv(file_path)
    
    if df.empty: 
        continue
    
    parts = os.path.normpath(file_path).split(os.sep)
    
    # On vérifie qu'on a assez de profondeur (au moins 4 éléments : le fichier + 3 dossiers)
    if len(parts) >= 4:
        domain = parts[ -4 ]
        
        # Déduction de la Source et de la Target selon le nom du domaine
        if "_to_" in domain:
            domain_source, domain_target = domain.split("_to_", 1)
        else:
            domain_source = domain
            domain_target = ""
            
        # Insertion des métadonnées contextuelles au début du DataFrame
        df.insert(0, 'Domain_Target', domain_target)
        df.insert(0, 'Domain_Source', domain_source)
        df.insert(0, 'Domain', domain)
    else:
        df.insert(0, 'Source_Path', os.path.dirname(file_path))
    
    df_list.append(df)
        

assert df_list, "Aucun DataFrame valide n'a été créé à partir des fichiers CSV"

print("🔗 Concaténation des DataFrames...")
master_df = pd.concat(df_list, ignore_index=True)

# output_csv_path = os.path.join(input_dir, 'master_radiomics.csv')
# master_df.to_csv(output_csv_path, index=False)

print("\n" + "="*50)
print("✅ FUSION TERMINÉE AVEC SUCCÈS")
print("="*50)
print(f"Total des lignes  : {len(master_df)}")
print(f"Total des colonnes: {len(master_df.columns)}")
print("="*50)

master_df.head()

📄 1796 fichiers trouvés. Début de la lecture...


Lecture ...: 100%|██████████| 1796/1796 [00:05<00:00, 314.51it/s]


🔗 Concaténation des DataFrames...

✅ FUSION TERMINÉE AVEC SUCCÈS
Total des lignes  : 7992
Total des colonnes: 101


,Domain,Domain_Source,Domain_Target,Subject_ID,VOI,Modality,Image_Filename,ROI_type,original_firstorder_10Percentile,original_firstorder_90Percentile,...,original_gldm_LargeDependenceLowGrayLevelEmphasis,original_gldm_LowGrayLevelEmphasis,original_gldm_SmallDependenceEmphasis,original_gldm_SmallDependenceHighGrayLevelEmphasis,original_gldm_SmallDependenceLowGrayLevelEmphasis,original_ngtdm_Busyness,original_ngtdm_Coarseness,original_ngtdm_Complexity,original_ngtdm_Contrast,original_ngtdm_Strength
0,nantes,nantes,,15080000,spleen,earl2,earl2.nii.gz,Original,1.372068,1.889349,...,11.969992,0.044853,0.009562,0.251885,0.000763,32.632969,0.000363,16.882386,0.004629,0.026981
1,nantes,nantes,,15080000,spleen,gaussian-earl2,gaussian-earl2.nii.gz,Original,1.352235,1.820446,...,17.316144,0.045244,0.006455,0.165800,0.000516,25.795765,0.000520,9.320394,0.003386,0.028230
2,nantes,nantes,,15080000,spleen,standard,pet.nii.gz,Original,1.333326,1.968653,...,7.311743,0.047045,0.016302,0.481772,0.001227,33.411213,0.000335,27.895862,0.007276,0.027515
3,nantes,nantes,,15080000,spleen,pseudo-earl2,pseudo-earl2.nii.gz,Original,1.371308,1.888349,...,11.910279,0.044909,0.009604,0.248926,0.000802,32.753831,0.000362,16.963075,0.004646,0.026868
4,nantes,nantes,,15080000,brain,earl2,earl2.nii.gz,Sphere_20.0mm,3.058370,11.937186,...,2.065838,0.032133,0.256412,172.043503,0.002071,0.185523,0.007615,1525.310888,0.172210,9.819384


### in case we load a presaved file

In [ ]:
import os
import pandas as pd

master_df = pd.read_csv('../data/master_radiomics.csv', low_memory=False)
master_df

In [172]:
import os
import numpy as np
import pandas as pd
import SimpleITK as sitk
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm
import logging
import warnings

from skimage.metrics import structural_similarity as ssim
from scipy.stats import wilcoxon, spearmanr
from scipy.ndimage import distance_transform_edt

# Désactiver les warnings
sitk.ProcessObject.SetGlobalWarningDisplay(False)
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

def generate_centered_sphere(sitk_mask, radius_mm=20.0, use_barycenter=False, margin_mm=1.0, shift_mm=(0.0, 0.0, 0.0)):
    spacing = sitk_mask.GetSpacing() 
    size = sitk_mask.GetSize()
    
    stats = sitk.LabelShapeStatisticsImageFilter()
    sitk_mask_uint8 = sitk.Cast(sitk_mask, sitk.sitkUInt8)
    stats.Execute(sitk_mask_uint8)
    
    if not stats.HasLabel(1):
        return None, "Masque vide"
        
    if use_barycenter:
        c_phys = list(stats.GetCentroid(1))
    else:
        mask_arr = sitk.GetArrayFromImage(sitk_mask_uint8)
        edt_map = distance_transform_edt(mask_arr, sampling=spacing[ ::-1 ])
        max_idx_flat = edt_map.argmax()
        cz, cy, cx = np.unravel_index(max_idx_flat, mask_arr.shape)
        c_phys = list(sitk_mask.TransformIndexToPhysicalPoint((int(cx), int(cy), int(cz))))

    c_phys[ 0 ] += shift_mm[ 0 ]
    c_phys[ 1 ] += shift_mm[ 1 ]
    c_phys[ 2 ] += shift_mm[ 2 ]
    
    c_idx = sitk_mask.TransformPhysicalPointToContinuousIndex(c_phys)
    
    nz, ny, nx = size[ 2 ], size[ 1 ], size[ 0 ]
    zz, yy, xx = np.ogrid[ :nz, :ny, :nx ]
    
    dist2 = (
        ((xx - c_idx[ 0 ]) * spacing[ 0 ])**2 + 
        ((yy - c_idx[ 1 ]) * spacing[ 1 ])**2 + 
        ((zz - c_idx[ 2 ]) * spacing[ 2 ])**2
    )
    
    sphere_arr = (dist2 <= radius_mm**2)
    safety_radius_sq = (radius_mm + margin_mm)**2
    safety_arr = (dist2 <= safety_radius_sq)

    if not np.any(sphere_arr):
        return None, "Sphère vide"

    mask_arr_bool = sitk.GetArrayFromImage(sitk_mask_uint8).astype(bool)

    if not np.all(mask_arr_bool[ safety_arr ]):
        return None, "Débordement"

    out_sitk = sitk.GetImageFromArray(sphere_arr.astype(np.uint8))
    out_sitk.CopyInformation(sitk_mask)
    return out_sitk, "Succès"

def calculate_metrics(gt_arr, pred_arr, mask_arr):
    voxels_gt = gt_arr[ mask_arr > 0 ]
    voxels_pred = pred_arr[ mask_arr > 0 ]
    
    if len(voxels_gt) == 0:
        return None
        
    # --- aRE ---
    voxels_gt_safe = np.where(voxels_gt == 0, 1e-8, voxels_gt)
    are_val = np.mean(np.abs((voxels_pred - voxels_gt_safe) / voxels_gt_safe)) * 100
    
    # --- PSNR ---
    mse = np.mean((voxels_gt - voxels_pred) ** 2)
    max_val = np.max(voxels_gt)
    psnr_val = 10 * np.log10((max_val ** 2) / mse) if mse > 1e-8 else np.inf
    
    # --- COV (Coefficient of Variation) ---
    mean_gt, mean_pred = np.mean(voxels_gt), np.mean(voxels_pred)
    cov_gt = (np.std(voxels_gt) / mean_gt) if mean_gt > 1e-8 else 0
    cov_pred = (np.std(voxels_pred) / mean_pred) if mean_pred > 1e-8 else 0
    
    # --- SSIM (3D) ---
    # Crop sur la Bounding Box du masque pour éviter l'influence massive du fond
    z, y, x = np.where(mask_arr > 0)
    z_min, z_max = np.min(z), np.max(z)
    y_min, y_max = np.min(y), np.max(y)
    x_min, x_max = np.min(x), np.max(x)
    
    gt_box = gt_arr[ z_min:z_max+1, y_min:y_max+1, x_min:x_max+1 ].copy()
    pred_box = pred_arr[ z_min:z_max+1, y_min:y_max+1, x_min:x_max+1 ].copy()
    mask_box = mask_arr[ z_min:z_max+1, y_min:y_max+1, x_min:x_max+1 ]
    
    # On met strictement à zéro ce qui est hors du masque dans la bounding box
    gt_box[ mask_box == 0 ] = 0
    pred_box[ mask_box == 0 ] = 0
    
    data_range = float(np.max(gt_box) - np.min(gt_box))
    win_size = min(7, min(gt_box.shape))
    if win_size % 2 == 0: 
        win_size -= 1
        
    if win_size >= 3:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            ssim_val = ssim(gt_box, pred_box, data_range=data_range, win_size=win_size)
    else:
        ssim_val = np.nan # La zone est trop petite pour calculer une texture SSIM
        
    return {
        "aRE": are_val,
        "PSNR": psnr_val,
        "SSIM": ssim_val,
        "COV_GT": cov_gt,
        "COV_Pred": cov_pred,
        "Mean_GT": mean_gt,
        "Mean_Pred": mean_pred
    }

def process_single_subject(args):
    subject_id, subject_path, gt_filename, pred_filename, mask_filename, sphere_vois, sphere_radius = args
    results_dict = {}
    
    if not os.path.isdir(subject_path):
        return subject_id, None, "Invalid path"
        
    # Les dossiers VOI sont situés directement dans le dossier du patient
    voi_folders = [f for f in os.listdir(subject_path) if os.path.isdir(os.path.join(subject_path, f))]
    
    for voi in voi_folders:
        voi_path = os.path.join(subject_path, voi)
        files = os.listdir(voi_path)
        
        gt_file = gt_filename if gt_filename.endswith('nii.gz') else gt_filename + ".nii.gz"
        pred_file = pred_filename if pred_filename.endswith('nii.gz') else pred_filename + ".nii.gz"
        
        if not gt_file or not pred_file or mask_filename not in files:
            continue
            
        try:
            gt_img = sitk.ReadImage(os.path.join(voi_path, gt_file))
            pred_img = sitk.ReadImage(os.path.join(voi_path, pred_file))
            mask_img = sitk.ReadImage(os.path.join(voi_path, mask_filename))
            
            # --- Condition Sphère vs Organe entier ---
            if voi in sphere_vois:
                sphere_img, msg = generate_centered_sphere(mask_img, radius_mm=sphere_radius)
                if sphere_img is not None:
                    mask_img = sphere_img
                else:
                    continue # La sphère est invalide (déborde ou vide), on ignore ce VOI
                    
            gt_arr = sitk.GetArrayFromImage(gt_img)
            pred_arr = sitk.GetArrayFromImage(pred_img)
            mask_arr = sitk.GetArrayFromImage(mask_img)
            
            metrics = calculate_metrics(gt_arr, pred_arr, mask_arr)
            if metrics:
                results_dict[ voi ] = metrics
                
        except Exception as e:
            pass # Poursuite silencieuse pour ne pas crasher le pool entier
            
    return subject_id, results_dict, "Success"


def evaluate(
    root_dir, include_only=None, sphere_vois=None, sphere_radius=20.0, 
    mask_filename="mask.nii.gz", gt_filename="EARL", pred_filename="predicted", num_workers=None
    ):
    if sphere_vois is None:
        sphere_vois = []
        
    if not os.path.exists(root_dir):
        logging.error(f"Le dossier {root_dir} n'existe pas.")
        return {}

    subjects = [s for s in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, s))]
    if include_only:
        subjects = [s for s in subjects if s in include_only]
    
    tasks = [
        (subj, os.path.join(root_dir, subj), gt_filename, pred_filename, mask_filename, sphere_vois, sphere_radius) 
        for subj in subjects
    ]
    
    center_results = {}
    print(f"\n🚀 Lancement de l'évaluation sur : {os.path.basename(root_dir)} ({len(subjects)} patients)")
    print(f"🎯 Organes évalués par sphère ({sphere_radius}mm) : {sphere_vois}")
    
    with ProcessPoolExecutor(max_workers=num_workers) as executor:
        futures = {executor.submit(process_single_subject, task): task for task in tasks}
        
        for future in tqdm(as_completed(futures), total=len(tasks), desc="Calcul des Métriques"):
            subject_id, subj_results, status = future.result()
            
            if subj_results:
                center_results[ subject_id ] = subj_results

    return center_results

def display_statistics(res):
    """
    Parcourt les résultats par patient, regroupe par VOI et affiche les statistiques + p-values.
    """
    # 1. Regrouper les résultats par VOI
    vois_trouvees = set()
    for subj in res.values():
        vois_trouvees.update(subj.keys())
        
    for voi in sorted(vois_trouvees):
        print("\n" + "="*60)
        print(f"📊 RÉSULTATS POUR LA VOI : {voi.upper()}")
        print("="*60)
        
        are_list, psnr_list, ssim_list = [], [], []
        cov_gt_list, cov_pred_list = [], []
        mean_gt_list, mean_pred_list = [], []
        
        for subj_id, subj_data in res.items():
            if voi in subj_data:
                m = subj_data[ voi ]
                if not np.isnan(m[ "aRE" ]): are_list.append(m[ "aRE" ])
                if not np.isnan(m[ "PSNR" ]) and m[ "PSNR" ] != np.inf: psnr_list.append(m[ "PSNR" ])
                if not np.isnan(m[ "SSIM" ]): ssim_list.append(m[ "SSIM" ])
                
                cov_gt_list.append(m[ "COV_GT" ])
                cov_pred_list.append(m[ "COV_Pred" ])
                mean_gt_list.append(m[ "Mean_GT" ])
                mean_pred_list.append(m[ "Mean_Pred" ])
        
        n = len(are_list)
        if n == 0:
            print("Aucune donnée valide pour ce calcul.")
            continue
            
        print(f"🔹 aRE  : {np.mean(are_list):.2f}% ± {np.std(are_list):.2f}%")
        print(f"🔹 PSNR : {np.mean(psnr_list):.2f} ± {np.std(psnr_list):.2f} dB")
        print(f"🔹 SSIM : {np.mean(ssim_list):.4f} ± {np.std(ssim_list):.4f}")
        print(f"🔹 COV  : GT = {np.mean(cov_gt_list):.3f} | Pred = {np.mean(cov_pred_list):.3f}")
        
        # --- Tests Statistiques ---
        try:
            _, p_mean = wilcoxon(mean_gt_list, mean_pred_list)
        except ValueError:
            p_mean = np.nan # Échoue si les arrays sont strictement identiques
            
        try:
            _, p_cov = wilcoxon(cov_gt_list, cov_pred_list)
        except ValueError:
            p_cov = np.nan
            
        corr, p_corr = spearmanr(mean_gt_list, mean_pred_list)
        
        print(f"\n🧪 Tests Statistiques (N = {n} patients valides)")
        print(f"  - Biais d'Intensité (Wilcoxon sur moyennes GT/Pred) : p-value = {p_mean:.4e}")
        print(f"  - Biais de Variance (Wilcoxon sur COV GT/Pred)      : p-value = {p_cov:.4e}")
        print(f"  - Préservation du rang patient (Spearman ρ)         : ρ = {corr:.3f} (p = {p_corr:.4e})")

In [174]:
# Lancement des calculs
resultats = evaluate(
    root_dir='../outputs/pseudo-earl/a100/',
    sphere_vois=['liver', 'lung', 'brain'],  # Celles-ci seront en sphère de 20mm
    mask_filename='mask.nii.gz',             # Les autres, comme 'spleen', utiliseront le masque entier
    gt_filename='earl1', 
    pred_filename='pseudo-earl1'
)

display_statistics(resultats)


🚀 Lancement de l'évaluation sur :  (50 patients)
🎯 Organes évalués par sphère (20.0mm) : ['liver', 'lung', 'brain']


Calcul des Métriques: 100%|██████████| 50/50 [00:03<00:00, 12.75it/s]


📊 RÉSULTATS POUR LA VOI : BRAIN
🔹 aRE  : 0.67% ± 0.15%
🔹 PSNR : 48.12 ± 1.27 dB
🔹 SSIM : 0.9998 ± 0.0001
🔹 COV  : GT = 0.446 | Pred = 0.446

🧪 Tests Statistiques (N = 50 patients valides)
  - Biais d'Intensité (Wilcoxon sur moyennes GT/Pred) : p-value = 4.3169e-03
  - Biais de Variance (Wilcoxon sur COV GT/Pred)      : p-value = 2.1866e-01
  - Préservation du rang patient (Spearman ρ)         : ρ = 1.000 (p = 2.0511e-79)

📊 RÉSULTATS POUR LA VOI : LESION
🔹 aRE  : 1.02% ± 0.49%
🔹 PSNR : 42.88 ± 4.20 dB
🔹 SSIM : 0.9998 ± 0.0006
🔹 COV  : GT = 0.250 | Pred = 0.251

🧪 Tests Statistiques (N = 39 patients valides)
  - Biais d'Intensité (Wilcoxon sur moyennes GT/Pred) : p-value = 2.1948e-01
  - Biais de Variance (Wilcoxon sur COV GT/Pred)      : p-value = 3.4665e-02
  - Préservation du rang patient (Spearman ρ)         : ρ = 1.000 (p = 8.2719e-59)

📊 RÉSULTATS POUR LA VOI : LIVER
🔹 aRE  : 0.42% ± 0.05%
🔹 PSNR : 46.48 ± 1.08 dB
🔹 SSIM : 0.9986 ± 0.0004
🔹 COV  : GT = 0.049 | Pred = 0.049

🧪 Tes

In [175]:
# Lancement des calculs
resultats = evaluate(
    root_dir='../outputs/pseudo-earl/a100/',
    sphere_vois=['liver', 'lung', 'brain'],  # Celles-ci seront en sphère de 20mm
    mask_filename='mask.nii.gz',             # Les autres, comme 'spleen', utiliseront le masque entier
    gt_filename='earl2', 
    pred_filename='pseudo-earl2'
)

display_statistics(resultats)


🚀 Lancement de l'évaluation sur :  (50 patients)
🎯 Organes évalués par sphère (20.0mm) : ['liver', 'lung', 'brain']


Calcul des Métriques: 100%|██████████| 50/50 [00:03<00:00, 13.32it/s]


📊 RÉSULTATS POUR LA VOI : BRAIN
🔹 aRE  : 1.13% ± 0.32%
🔹 PSNR : 45.13 ± 1.86 dB
🔹 SSIM : 0.9996 ± 0.0003
🔹 COV  : GT = 0.502 | Pred = 0.503

🧪 Tests Statistiques (N = 50 patients valides)
  - Biais d'Intensité (Wilcoxon sur moyennes GT/Pred) : p-value = 4.1932e-02
  - Biais de Variance (Wilcoxon sur COV GT/Pred)      : p-value = 1.2822e-03
  - Préservation du rang patient (Spearman ρ)         : ρ = 1.000 (p = 2.0420e-76)

📊 RÉSULTATS POUR LA VOI : LESION
🔹 aRE  : 1.71% ± 0.63%
🔹 PSNR : 39.98 ± 4.60 dB
🔹 SSIM : 0.9996 ± 0.0011
🔹 COV  : GT = 0.304 | Pred = 0.305

🧪 Tests Statistiques (N = 39 patients valides)
  - Biais d'Intensité (Wilcoxon sur moyennes GT/Pred) : p-value = 3.9491e-01
  - Biais de Variance (Wilcoxon sur COV GT/Pred)      : p-value = 2.8290e-01
  - Préservation du rang patient (Spearman ρ)         : ρ = 0.998 (p = 2.0235e-44)

📊 RÉSULTATS POUR LA VOI : LIVER
🔹 aRE  : 0.99% ± 0.20%
🔹 PSNR : 40.69 ± 1.52 dB
🔹 SSIM : 0.9974 ± 0.0008
🔹 COV  : GT = 0.093 | Pred = 0.092

🧪 Tes

In [176]:
# Lancement des calculs
resultats = evaluate(
    root_dir='../outputs/pseudo-earl/chb/',
    sphere_vois=['liver', 'lung', 'brain'],  # Celles-ci seront en sphère de 20mm
    mask_filename='mask.nii.gz',             # Les autres, comme 'spleen', utiliseront le masque entier
    gt_filename='earl1', 
    pred_filename='pseudo-earl1'
)

display_statistics(resultats)


🚀 Lancement de l'évaluation sur :  (100 patients)
🎯 Organes évalués par sphère (20.0mm) : ['liver', 'lung', 'brain']


Calcul des Métriques: 100%|██████████| 100/100 [00:05<00:00, 18.75it/s]


📊 RÉSULTATS POUR LA VOI : BRAIN
🔹 aRE  : 0.88% ± 0.86%
🔹 PSNR : 45.31 ± 6.59 dB
🔹 SSIM : 0.9993 ± 0.0041
🔹 COV  : GT = 0.347 | Pred = 0.349

🧪 Tests Statistiques (N = 99 patients valides)
  - Biais d'Intensité (Wilcoxon sur moyennes GT/Pred) : p-value = 6.9031e-03
  - Biais de Variance (Wilcoxon sur COV GT/Pred)      : p-value = 2.4414e-07
  - Préservation du rang patient (Spearman ρ)         : ρ = 1.000 (p = 3.8226e-162)

📊 RÉSULTATS POUR LA VOI : LESION
🔹 aRE  : 0.34% ± 0.30%
🔹 PSNR : 54.16 ± 5.43 dB
🔹 SSIM : 1.0000 ± 0.0000
🔹 COV  : GT = 0.306 | Pred = 0.306

🧪 Tests Statistiques (N = 82 patients valides)
  - Biais d'Intensité (Wilcoxon sur moyennes GT/Pred) : p-value = 6.8584e-01
  - Biais de Variance (Wilcoxon sur COV GT/Pred)      : p-value = 6.9606e-01
  - Préservation du rang patient (Spearman ρ)         : ρ = 1.000 (p = 4.6167e-128)

📊 RÉSULTATS POUR LA VOI : LIVER
🔹 aRE  : 0.27% ± 0.08%
🔹 PSNR : 51.78 ± 2.14 dB
🔹 SSIM : 0.9998 ± 0.0001
🔹 COV  : GT = 0.098 | Pred = 0.098

🧪 T

In [177]:
# Lancement des calculs
resultats = evaluate(
    root_dir='../outputs/pseudo-earl/rennes/',
    sphere_vois=['liver', 'lung', 'brain'],  # Celles-ci seront en sphère de 20mm
    mask_filename='mask.nii.gz',             # Les autres, comme 'spleen', utiliseront le masque entier
    gt_filename='earl1', 
    pred_filename='pseudo-earl1'
)

display_statistics(resultats)


🚀 Lancement de l'évaluation sur :  (75 patients)
🎯 Organes évalués par sphère (20.0mm) : ['liver', 'lung', 'brain']


Calcul des Métriques: 100%|██████████| 75/75 [00:04<00:00, 16.50it/s]


📊 RÉSULTATS POUR LA VOI : BRAIN
🔹 aRE  : 0.87% ± 0.19%
🔹 PSNR : 45.76 ± 1.68 dB
🔹 SSIM : 0.9997 ± 0.0002
🔹 COV  : GT = 0.452 | Pred = 0.453

🧪 Tests Statistiques (N = 65 patients valides)
  - Biais d'Intensité (Wilcoxon sur moyennes GT/Pred) : p-value = 1.1762e-02
  - Biais de Variance (Wilcoxon sur COV GT/Pred)      : p-value = 4.1222e-04
  - Préservation du rang patient (Spearman ρ)         : ρ = 1.000 (p = 1.5006e-107)

📊 RÉSULTATS POUR LA VOI : LESION
🔹 aRE  : 2.57% ± 1.40%
🔹 PSNR : 36.09 ± 4.81 dB
🔹 SSIM : 0.9995 ± 0.0015
🔹 COV  : GT = 0.314 | Pred = 0.314

🧪 Tests Statistiques (N = 53 patients valides)
  - Biais d'Intensité (Wilcoxon sur moyennes GT/Pred) : p-value = 9.6822e-01
  - Biais de Variance (Wilcoxon sur COV GT/Pred)      : p-value = 7.6680e-01
  - Préservation du rang patient (Spearman ρ)         : ρ = 0.999 (p = 7.1190e-71)

📊 RÉSULTATS POUR LA VOI : LIVER
🔹 aRE  : 1.41% ± 0.27%
🔹 PSNR : 37.91 ± 1.43 dB
🔹 SSIM : 0.9957 ± 0.0014
🔹 COV  : GT = 0.109 | Pred = 0.107

🧪 Te

In [178]:
# Lancement des calculs
resultats = evaluate(
    root_dir='../outputs/pseudo-earl/nantes/',
    sphere_vois=['liver', 'lung', 'brain'],  # Celles-ci seront en sphère de 20mm
    mask_filename='mask.nii.gz',             # Les autres, comme 'spleen', utiliseront le masque entier
    gt_filename='earl2', 
    pred_filename='pseudo-earl2'
)

display_statistics(resultats)


🚀 Lancement de l'évaluation sur :  (88 patients)
🎯 Organes évalués par sphère (20.0mm) : ['liver', 'lung', 'brain']


Calcul des Métriques: 100%|██████████| 88/88 [00:05<00:00, 16.76it/s]


📊 RÉSULTATS POUR LA VOI : BRAIN
🔹 aRE  : 0.30% ± 0.07%
🔹 PSNR : 53.78 ± 1.90 dB
🔹 SSIM : 0.9999 ± 0.0000
🔹 COV  : GT = 0.540 | Pred = 0.540

🧪 Tests Statistiques (N = 88 patients valides)
  - Biais d'Intensité (Wilcoxon sur moyennes GT/Pred) : p-value = 2.1500e-01
  - Biais de Variance (Wilcoxon sur COV GT/Pred)      : p-value = 5.9822e-06
  - Préservation du rang patient (Spearman ρ)         : ρ = 1.000 (p = 9.1707e-173)

📊 RÉSULTATS POUR LA VOI : LESION
🔹 aRE  : 0.45% ± 0.31%
🔹 PSNR : 50.05 ± 4.93 dB
🔹 SSIM : 1.0000 ± 0.0001
🔹 COV  : GT = 0.310 | Pred = 0.310

🧪 Tests Statistiques (N = 69 patients valides)
  - Biais d'Intensité (Wilcoxon sur moyennes GT/Pred) : p-value = 4.7863e-01
  - Biais de Variance (Wilcoxon sur COV GT/Pred)      : p-value = 7.4620e-05
  - Préservation du rang patient (Spearman ρ)         : ρ = 1.000 (p = 3.9002e-120)

📊 RÉSULTATS POUR LA VOI : LIVER
🔹 aRE  : 0.23% ± 0.04%
🔹 PSNR : 50.94 ± 1.10 dB
🔹 SSIM : 0.9997 ± 0.0001
🔹 COV  : GT = 0.083 | Pred = 0.083

🧪 T

### Radiomics evaluation

In [179]:
import numpy as np
import pandas as pd
from scipy.stats import norm, ttest_1samp
import warnings

# =============================================================================
# 1. FONCTIONS MATHÉMATIQUES (AVEC P-VALUES INTÉGRÉES)
# =============================================================================

def calculate_ccc_and_ci(y_true, y_pred, alpha=0.05):
    y_true = np.asarray(y_true, dtype=float).flatten()
    y_pred = np.asarray(y_pred, dtype=float).flatten()
    N = len(y_true)
    
    if N < 3: 
        return np.nan, np.nan, np.nan, np.nan
    
    if np.var(y_true) == 0 and np.var(y_pred) == 0:
        return 1.0, 1.0, 1.0, 0.0
        
    cor = np.corrcoef(y_true, y_pred)[0, 1]
    if np.isnan(cor): 
        return np.nan, np.nan, np.nan, np.nan
    
    mean_true, mean_pred = np.mean(y_true), np.mean(y_pred)
    var_true, var_pred = np.var(y_true), np.var(y_pred)
    sd_true, sd_pred = np.std(y_true), np.std(y_pred)
    
    numerator = 2 * cor * sd_true * sd_pred
    denominator = var_true + var_pred + (mean_true - mean_pred)**2
    ccc = numerator / denominator if denominator != 0 else np.nan
    
    if np.isnan(ccc):
        return np.nan, np.nan, np.nan, np.nan
        
    ccc = np.clip(ccc, -1.0, 1.0)
    
    if ccc == 1.0:
        return 1.0, 1.0, 1.0, 0.0 

    z = np.arctanh(ccc)
    se = 1.0 / np.sqrt(N - 2)
    z_critical = norm.ppf(1 - alpha / 2) 
    
    lower_ci = np.tanh(z - z_critical * se)
    upper_ci = np.tanh(z + z_critical * se)
    
    z_stat = z / se
    p_value = 2 * (1 - norm.cdf(np.abs(z_stat)))
    
    return ccc, lower_ci, upper_ci, p_value

def format_p_value(p):
    """Formate la p-value pour la publication."""
    if np.isnan(p):
        return "NaN"
    elif p < 0.001:
        return "< 0.001"
    elif p < 0.01:
        return "< 0.01"
    elif p < 0.05:
        return "< 0.05"
    else:
        return f"{p:.3f}"

def calculate_relative_bland_altman(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float).flatten()
    y_pred = np.asarray(y_pred, dtype=float).flatten()
    
    denominator = (y_true + y_pred) / 2.0
    safe_mask = denominator != 0
    
    if not np.any(safe_mask):
        return 0.0, 0.0, 0.0, 0.0, np.nan

    # Calcul en pourcentage dès le départ
    relative_diffs_pct = ((y_pred[safe_mask] - y_true[safe_mask]) / denominator[safe_mask]) * 100.0
    
    bias_relative_pct = np.mean(relative_diffs_pct)
    std_relative_pct = np.std(relative_diffs_pct, ddof=1) if len(relative_diffs_pct) > 1 else 0

    loa_lower_pct = bias_relative_pct - (1.96 * std_relative_pct)
    loa_upper_pct = bias_relative_pct + (1.96 * std_relative_pct)
    
    if len(relative_diffs_pct) > 1 and std_relative_pct > 0:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            _, p_value = ttest_1samp(relative_diffs_pct, 0.0)
    else:
        p_value = np.nan
        
    return bias_relative_pct, std_relative_pct, loa_lower_pct, loa_upper_pct, p_value

# =============================================================================
# 2. ÉVALUATION PAR VOI DEPUIS LE MASTER DATAFRAME
# =============================================================================

def evaluate_radiomics_from_master(df, gt_mod='standard', pred_mod='harmonized'):
    df_gt = df[ df[ 'Modality' ] == gt_mod ].copy()
    df_pred = df[ df[ 'Modality' ] == pred_mod ].copy()
    
    merged = pd.merge(
        df_gt, df_pred, 
        on=[ 'Subject_ID', 'VOI', 'Domain_Source' ], 
        suffixes=('_gt', '_pred')
    )
    
    if merged.empty:
        print("⚠️ Aucune donnée appariée trouvée.")
        return None
        
    features = [c for c in df.columns if c.startswith('original_') and not c.startswith('original_shape_')]
    vois_list = merged[ 'VOI' ].unique()
    
    results_by_voi = {}
    
    for voi in vois_list:
        df_voi = merged[ merged[ 'VOI' ] == voi ]
        metrics_list = []
        
        for feat in features:
            gt_vals = df_voi[ f"{feat}_gt" ].dropna().values
            pred_vals = df_voi[ f"{feat}_pred" ].dropna().values
            
            if len(gt_vals) < 3:
                continue
                
            ccc, ccc_ci_low, ccc_ci_up, p_ccc = calculate_ccc_and_ci(gt_vals, pred_vals)
            bias_pct, std_diff_pct, loa_inf_pct, loa_sup_pct, p_bias = calculate_relative_bland_altman(gt_vals, pred_vals)
            
            family = feat.split('_')[ 1 ]
            
            metrics_list.append({
                'Feature': feat,
                'Family': family,
                'CCC': ccc,
                'CCC_95CI_Lower': ccc_ci_low,
                'CCC_95CI_Upper': ccc_ci_up,
                'p_val_CCC': p_ccc,
                'Bias_%': bias_pct,          
                'Bias_STD_%': std_diff_pct,    
                'LoA_Lower_%': loa_inf_pct,    
                'LoA_Upper_%': loa_sup_pct,
                'p_val_Bias': p_bias
            })
            
        if not metrics_list:
            continue
            
        df_metrics = pd.DataFrame(metrics_list).dropna(subset=['CCC'])
        
        family_summary = df_metrics.groupby('Family').agg({
            'CCC': 'mean',
            'CCC_95CI_Lower': 'mean',
            'CCC_95CI_Upper': 'mean',
            'Bias_%': 'mean',
            'Bias_STD_%': 'mean', # Agrégation de l'écart type
            'LoA_Lower_%': 'mean',
            'LoA_Upper_%': 'mean',
        })
        
        # --- NOUVEAU FORMATAGE DU TABLEAU ---
        family_summary[ 'CCC [95% CI]' ] = family_summary.apply(
            lambda x: f"{x[ 'CCC' ]:.3f} [{x[ 'CCC_95CI_Lower' ]:.3f}, {x[ 'CCC_95CI_Upper' ]:.3f}]", axis=1
        )
        family_summary[ 'Bias (%)' ] = family_summary[ 'Bias_%' ].apply(lambda x: f"{x:.2f}%")
        family_summary[ 'SD (%)' ] = family_summary[ 'Bias_STD_%' ].apply(lambda x: f"{x:.2f}%")
        family_summary[ '95% LoA (%)' ] = family_summary.apply(
            lambda x: f"[{x[ 'LoA_Lower_%' ]:.2f}%, {x[ 'LoA_Upper_%' ]:.2f}%]", axis=1
        )
        
        # Résumé Global pour la VOI
        voi_global = {
            'N_Pairs': len(df_voi),
            'CCC_Mean': df_metrics[ 'CCC' ].mean(),
            'p_val_CCC_Median': df_metrics[ 'p_val_CCC' ].median(),
            'Bias_Mean': df_metrics[ 'Bias_%' ].mean(),
            'Bias_STD': df_metrics[ 'Bias_STD_%' ].mean(),
            'p_val_Bias_Median': df_metrics[ 'p_val_Bias' ].median(),
        }
        
        results_by_voi[ voi ] = {
            'df_details': df_metrics,
            'df_family': family_summary,
            'global': voi_global
        }
        
    return results_by_voi

# =============================================================================
# 3. AFFICHAGE (STYLE VOXEL-WISE + MOYENNE GLOBALE)
# =============================================================================

def display_radiomics_statistics(results_by_voi):
    if not results_by_voi:
        return
        
    grand_ccc = []
    grand_pval_ccc = []
    grand_bias = []
    grand_pval_bias = []
    
    for voi in sorted(results_by_voi.keys()):
        data = results_by_voi[ voi ]
        g = data[ 'global' ]
        fam = data[ 'df_family' ]
        
        print("\n" + "="*80)
        print(f"📊 RÉSULTATS POUR LA VOI : {voi.upper()}")
        print("="*80)
        
        p_ccc_str = format_p_value(g['p_val_CCC_Median'])
        p_bias_str = format_p_value(g['p_val_Bias_Median'])
        
        print(f"🔹 CCC Moyen Global   : {g['CCC_Mean']:.3f} (p {p_ccc_str})")
        print(f"🔹 Biais Relatif      : {g['Bias_Mean']:.2f}% ± {g['Bias_STD']:.2f}% (p {p_bias_str})")
        print(f"   (Testé sur N = {g['N_Pairs']} paires)")
        
        print("\n  RÉSULTATS PAR FAMILLE DE RADIOMIQUES :")
        
        # Nouvelles colonnes pour un affichage propre
        cols_to_show = [ 'CCC [95% CI]', 'Bias (%)', 'SD (%)', '95% LoA (%)' ]
        print(fam[ cols_to_show ].to_markdown())
        
        grand_ccc.append(g[ 'CCC_Mean' ])
        grand_pval_ccc.append(g[ 'p_val_CCC_Median' ])
        grand_bias.append(g[ 'Bias_Mean' ])
        grand_pval_bias.append(g[ 'p_val_Bias_Median' ])

    print("\n" + "="*80)
    print("🌐 RÉSULTAT GLOBAL (Toutes VOIs confondues)")
    print("="*80)
    
    global_p_ccc_str = format_p_value(np.nanmedian(grand_pval_ccc))
    global_p_bias_str = format_p_value(np.nanmedian(grand_pval_bias))
    
    print(f"🔹 CCC Moyen        : {np.nanmean(grand_ccc):.3f} (p {global_p_ccc_str})")
    print(f"🔹 Biais Relatif    : {np.nanmean(grand_bias):.2f}% (p {global_p_bias_str})")
    print("="*80)

In [182]:
rad_results = evaluate_radiomics_from_master(
    master_df[master_df['Domain'] == 'a100'], 
    gt_mod='earl1',
    pred_mod='pseudo-earl1'
)

display_radiomics_statistics(rad_results)


📊 RÉSULTATS POUR LA VOI : BRAIN
🔹 CCC Moyen Global   : 0.997 (p < 0.001)
🔹 Biais Relatif      : 0.11% ± 2.72% (p 0.209)
   (Testé sur N = 50 paires)

  RÉSULTATS PAR FAMILLE DE RADIOMIQUES :
| Family     | CCC [95% CI]         | Bias (%)   | SD (%)   | 95% LoA (%)       |
|:-----------|:---------------------|:-----------|:---------|:------------------|
| firstorder | 1.000 [1.000, 1.000] | -0.02%     | 0.57%    | [-1.13%, 1.09%]   |
| glcm       | 0.998 [0.996, 0.999] | 0.04%      | 0.80%    | [-1.52%, 1.60%]   |
| gldm       | 0.991 [0.984, 0.995] | 0.32%      | 4.16%    | [-7.84%, 8.48%]   |
| glrlm      | 0.997 [0.994, 0.998] | 0.21%      | 2.95%    | [-5.56%, 5.99%]   |
| glszm      | 0.995 [0.992, 0.997] | 0.06%      | 6.32%    | [-12.33%, 12.46%] |
| ngtdm      | 0.998 [0.996, 0.999] | 0.22%      | 3.36%    | [-6.36%, 6.81%]   |

📊 RÉSULTATS POUR LA VOI : LESION
🔹 CCC Moyen Global   : 0.992 (p < 0.001)
🔹 Biais Relatif      : 0.79% ± 16.37% (p 0.164)
   (Testé sur N = 39 paires)


In [183]:
rad_results = evaluate_radiomics_from_master(
    master_df[master_df['Domain'] == 'a100'], 
    gt_mod='earl2',
    pred_mod='pseudo-earl2'
)

display_radiomics_statistics(rad_results)


📊 RÉSULTATS POUR LA VOI : BRAIN
🔹 CCC Moyen Global   : 0.989 (p < 0.001)
🔹 Biais Relatif      : 0.20% ± 4.01% (p 0.286)
   (Testé sur N = 50 paires)

  RÉSULTATS PAR FAMILLE DE RADIOMIQUES :
| Family     | CCC [95% CI]         | Bias (%)   | SD (%)   | 95% LoA (%)       |
|:-----------|:---------------------|:-----------|:---------|:------------------|
| firstorder | 1.000 [1.000, 1.000] | 0.16%      | 0.69%    | [-1.19%, 1.51%]   |
| glcm       | 0.998 [0.996, 0.999] | 0.17%      | 1.76%    | [-3.29%, 3.63%]   |
| gldm       | 0.980 [0.966, 0.988] | 0.31%      | 6.50%    | [-12.43%, 13.04%] |
| glrlm      | 0.990 [0.983, 0.994] | 0.19%      | 5.07%    | [-9.74%, 10.12%]  |
| glszm      | 0.967 [0.948, 0.980] | 0.19%      | 8.04%    | [-15.56%, 15.94%] |
| ngtdm      | 0.997 [0.994, 0.998] | 0.28%      | 3.51%    | [-6.59%, 7.16%]   |

📊 RÉSULTATS POUR LA VOI : LESION
🔹 CCC Moyen Global   : 0.989 (p < 0.001)
🔹 Biais Relatif      : -0.10% ± 10.15% (p 0.445)
   (Testé sur N = 39 paires)

In [184]:
rad_results = evaluate_radiomics_from_master(
    master_df[master_df['Domain'] == 'chb'], 
    gt_mod='earl1',
    pred_mod='pseudo-earl1'
)

display_radiomics_statistics(rad_results)


📊 RÉSULTATS POUR LA VOI : BRAIN
🔹 CCC Moyen Global   : 0.989 (p < 0.001)
🔹 Biais Relatif      : 0.06% ± 7.49% (p < 0.01)
   (Testé sur N = 99 paires)

  RÉSULTATS PAR FAMILLE DE RADIOMIQUES :
| Family     | CCC [95% CI]         | Bias (%)   | SD (%)   | 95% LoA (%)       |
|:-----------|:---------------------|:-----------|:---------|:------------------|
| firstorder | 0.999 [0.998, 0.999] | 1.20%      | 11.81%   | [-21.95%, 24.34%] |
| glcm       | 0.996 [0.994, 0.997] | 0.41%      | 3.64%    | [-6.71%, 7.54%]   |
| gldm       | 0.983 [0.975, 0.988] | -0.09%     | 7.99%    | [-15.74%, 15.56%] |
| glrlm      | 0.992 [0.988, 0.994] | -0.05%     | 5.54%    | [-10.91%, 10.81%] |
| glszm      | 0.970 [0.956, 0.979] | -1.34%     | 10.60%   | [-22.12%, 19.44%] |
| ngtdm      | 0.992 [0.989, 0.995] | -0.43%     | 5.41%    | [-11.04%, 10.18%] |

📊 RÉSULTATS POUR LA VOI : LESION
🔹 CCC Moyen Global   : 0.987 (p < 0.001)
🔹 Biais Relatif      : -0.08% ± 8.02% (p 0.315)
   (Testé sur N = 82 paires)

In [185]:
rad_results = evaluate_radiomics_from_master(
    master_df[master_df['Domain'] == 'rennes'], 
    gt_mod='earl1',
    pred_mod='pseudo-earl1'
)

display_radiomics_statistics(rad_results)


📊 RÉSULTATS POUR LA VOI : BRAIN
🔹 CCC Moyen Global   : 0.986 (p < 0.001)
🔹 Biais Relatif      : 0.32% ± 3.83% (p 0.379)
   (Testé sur N = 65 paires)

  RÉSULTATS PAR FAMILLE DE RADIOMIQUES :
| Family     | CCC [95% CI]         | Bias (%)   | SD (%)   | 95% LoA (%)       |
|:-----------|:---------------------|:-----------|:---------|:------------------|
| firstorder | 1.000 [0.999, 1.000] | 0.01%      | 0.70%    | [-1.35%, 1.38%]   |
| glcm       | 0.997 [0.996, 0.998] | 0.08%      | 1.13%    | [-2.15%, 2.30%]   |
| gldm       | 0.987 [0.979, 0.992] | 0.53%      | 5.54%    | [-10.33%, 11.38%] |
| glrlm      | 0.994 [0.990, 0.996] | 0.36%      | 4.04%    | [-7.57%, 8.28%]   |
| glszm      | 0.941 [0.921, 0.958] | 0.88%      | 9.52%    | [-17.78%, 19.54%] |
| ngtdm      | 0.996 [0.994, 0.998] | 0.14%      | 4.43%    | [-8.55%, 8.83%]   |

📊 RÉSULTATS POUR LA VOI : LESION
🔹 CCC Moyen Global   : 0.947 (p < 0.001)
🔹 Biais Relatif      : 1.93% ± 24.61% (p 0.366)
   (Testé sur N = 53 paires)


In [186]:
rad_results = evaluate_radiomics_from_master(
    master_df[master_df['Domain'] == 'nantes'], 
    gt_mod='earl2',
    pred_mod='pseudo-earl2'
)

display_radiomics_statistics(rad_results)


📊 RÉSULTATS POUR LA VOI : BRAIN
🔹 CCC Moyen Global   : 0.998 (p < 0.001)
🔹 Biais Relatif      : -0.10% ± 1.72% (p < 0.05)
   (Testé sur N = 88 paires)

  RÉSULTATS PAR FAMILLE DE RADIOMIQUES :
| Family     | CCC [95% CI]         | Bias (%)   | SD (%)   | 95% LoA (%)     |
|:-----------|:---------------------|:-----------|:---------|:----------------|
| firstorder | 1.000 [1.000, 1.000] | 0.01%      | 0.20%    | [-0.39%, 0.42%] |
| glcm       | 1.000 [0.999, 1.000] | 0.03%      | 0.40%    | [-0.76%, 0.81%] |
| gldm       | 0.996 [0.994, 0.998] | -0.25%     | 2.84%    | [-5.82%, 5.32%] |
| glrlm      | 0.998 [0.998, 0.999] | -0.23%     | 2.07%    | [-4.28%, 3.82%] |
| glszm      | 0.995 [0.993, 0.997] | -0.27%     | 4.08%    | [-8.27%, 7.74%] |
| ngtdm      | 0.999 [0.998, 0.999] | 0.18%      | 1.63%    | [-3.03%, 3.38%] |

📊 RÉSULTATS POUR LA VOI : LESION
🔹 CCC Moyen Global   : 0.997 (p < 0.001)
🔹 Biais Relatif      : -0.21% ± 7.08% (p 0.136)
   (Testé sur N = 69 paires)

  RÉSULTATS P

## Gaussian EARL

In [188]:
resultats = evaluate(
    root_dir='../outputs/pseudo-earl/a100/',
    sphere_vois=['liver', 'lung', 'brain'],  # Celles-ci seront en sphère de 20mm
    mask_filename='mask.nii.gz',             # Les autres, comme 'spleen', utiliseront le masque entier
    gt_filename='earl1', 
    pred_filename='gaussian-earl1'
)

display_statistics(resultats)


🚀 Lancement de l'évaluation sur :  (50 patients)
🎯 Organes évalués par sphère (20.0mm) : ['liver', 'lung', 'brain']


Calcul des Métriques: 100%|██████████| 50/50 [00:03<00:00, 14.17it/s]


📊 RÉSULTATS POUR LA VOI : BRAIN
🔹 aRE  : 10.10% ± 2.37%
🔹 PSNR : 26.59 ± 0.89 dB
🔹 SSIM : 0.9762 ± 0.0047
🔹 COV  : GT = 0.446 | Pred = 0.362

🧪 Tests Statistiques (N = 50 patients valides)
  - Biais d'Intensité (Wilcoxon sur moyennes GT/Pred) : p-value = 5.1786e-02
  - Biais de Variance (Wilcoxon sur COV GT/Pred)      : p-value = 1.7764e-15
  - Préservation du rang patient (Spearman ρ)         : ρ = 0.999 (p = 2.3072e-63)

📊 RÉSULTATS POUR LA VOI : LESION
🔹 aRE  : 17.73% ± 7.67%
🔹 PSNR : 19.12 ± 5.27 dB
🔹 SSIM : 0.9641 ± 0.0579
🔹 COV  : GT = 0.250 | Pred = 0.213

🧪 Tests Statistiques (N = 39 patients valides)
  - Biais d'Intensité (Wilcoxon sur moyennes GT/Pred) : p-value = 3.6380e-12
  - Biais de Variance (Wilcoxon sur COV GT/Pred)      : p-value = 9.2041e-10
  - Préservation du rang patient (Spearman ρ)         : ρ = 0.981 (p = 6.0810e-28)

📊 RÉSULTATS POUR LA VOI : LIVER
🔹 aRE  : 1.68% ± 0.20%
🔹 PSNR : 34.84 ± 0.89 dB
🔹 SSIM : 0.9751 ± 0.0040
🔹 COV  : GT = 0.049 | Pred = 0.034

🧪 T

In [187]:
resultats = evaluate(
    root_dir='../outputs/pseudo-earl/a100/',
    sphere_vois=['liver', 'lung', 'brain'],  # Celles-ci seront en sphère de 20mm
    mask_filename='mask.nii.gz',             # Les autres, comme 'spleen', utiliseront le masque entier
    gt_filename='earl2', 
    pred_filename='gaussian-earl2'
)

display_statistics(resultats)


🚀 Lancement de l'évaluation sur :  (50 patients)
🎯 Organes évalués par sphère (20.0mm) : ['liver', 'lung', 'brain']


Calcul des Métriques: 100%|██████████| 50/50 [00:03<00:00, 12.64it/s]


📊 RÉSULTATS POUR LA VOI : BRAIN
🔹 aRE  : 2.39% ± 0.30%
🔹 PSNR : 38.58 ± 0.68 dB
🔹 SSIM : 0.9984 ± 0.0003
🔹 COV  : GT = 0.502 | Pred = 0.485

🧪 Tests Statistiques (N = 50 patients valides)
  - Biais d'Intensité (Wilcoxon sur moyennes GT/Pred) : p-value = 1.9588e-02
  - Biais de Variance (Wilcoxon sur COV GT/Pred)      : p-value = 1.7764e-15
  - Préservation du rang patient (Spearman ρ)         : ρ = 1.000 (p = 2.0420e-76)

📊 RÉSULTATS POUR LA VOI : LESION
🔹 aRE  : 4.00% ± 1.87%
🔹 PSNR : 32.19 ± 5.59 dB
🔹 SSIM : 0.9973 ± 0.0048
🔹 COV  : GT = 0.304 | Pred = 0.286

🧪 Tests Statistiques (N = 39 patients valides)
  - Biais d'Intensité (Wilcoxon sur moyennes GT/Pred) : p-value = 3.6380e-12
  - Biais de Variance (Wilcoxon sur COV GT/Pred)      : p-value = 7.2760e-12
  - Préservation du rang patient (Spearman ρ)         : ρ = 0.989 (p = 3.2098e-32)

📊 RÉSULTATS POUR LA VOI : LIVER
🔹 aRE  : 1.53% ± 0.17%
🔹 PSNR : 37.05 ± 0.73 dB
🔹 SSIM : 0.9933 ± 0.0007
🔹 COV  : GT = 0.093 | Pred = 0.078

🧪 Tes

In [189]:
resultats = evaluate(
    root_dir='../outputs/pseudo-earl/chb/',
    sphere_vois=['liver', 'lung', 'brain'],  # Celles-ci seront en sphère de 20mm
    mask_filename='mask.nii.gz',             # Les autres, comme 'spleen', utiliseront le masque entier
    gt_filename='earl1', 
    pred_filename='gaussian-earl1'
)

display_statistics(resultats)


🚀 Lancement de l'évaluation sur :  (100 patients)
🎯 Organes évalués par sphère (20.0mm) : ['liver', 'lung', 'brain']


Calcul des Métriques: 100%|██████████| 100/100 [00:04<00:00, 20.25it/s]


📊 RÉSULTATS POUR LA VOI : BRAIN
🔹 aRE  : 10.24% ± 3.15%
🔹 PSNR : 25.43 ± 1.27 dB
🔹 SSIM : 0.9609 ± 0.0126
🔹 COV  : GT = 0.347 | Pred = 0.264

🧪 Tests Statistiques (N = 99 patients valides)
  - Biais d'Intensité (Wilcoxon sur moyennes GT/Pred) : p-value = 1.3145e-06
  - Biais de Variance (Wilcoxon sur COV GT/Pred)      : p-value = 5.6972e-18
  - Préservation du rang patient (Spearman ρ)         : ρ = 0.998 (p = 1.0008e-123)

📊 RÉSULTATS POUR LA VOI : LESION
🔹 aRE  : 17.71% ± 9.35%
🔹 PSNR : 20.11 ± 5.88 dB
🔹 SSIM : 0.9669 ± 0.0558
🔹 COV  : GT = 0.306 | Pred = 0.248

🧪 Tests Statistiques (N = 82 patients valides)
  - Biais d'Intensité (Wilcoxon sur moyennes GT/Pred) : p-value = 3.6639e-15
  - Biais de Variance (Wilcoxon sur COV GT/Pred)      : p-value = 2.7060e-14
  - Préservation du rang patient (Spearman ρ)         : ρ = 0.925 (p = 1.9107e-35)

📊 RÉSULTATS POUR LA VOI : LIVER
🔹 aRE  : 4.31% ± 0.80%
🔹 PSNR : 28.12 ± 1.26 dB
🔹 SSIM : 0.9381 ± 0.0105
🔹 COV  : GT = 0.098 | Pred = 0.058

🧪 

In [190]:
resultats = evaluate(
    root_dir='../outputs/pseudo-earl/rennes/',
    sphere_vois=['liver', 'lung', 'brain'],  # Celles-ci seront en sphère de 20mm
    mask_filename='mask.nii.gz',             # Les autres, comme 'spleen', utiliseront le masque entier
    gt_filename='earl1', 
    pred_filename='gaussian-earl1'
)

display_statistics(resultats)


🚀 Lancement de l'évaluation sur :  (75 patients)
🎯 Organes évalués par sphère (20.0mm) : ['liver', 'lung', 'brain']


Calcul des Métriques: 100%|██████████| 75/75 [00:04<00:00, 15.53it/s]


📊 RÉSULTATS POUR LA VOI : BRAIN
🔹 aRE  : 6.39% ± 1.15%
🔹 PSNR : 29.91 ± 0.94 dB
🔹 SSIM : 0.9879 ± 0.0020
🔹 COV  : GT = 0.449 | Pred = 0.402

🧪 Tests Statistiques (N = 75 patients valides)
  - Biais d'Intensité (Wilcoxon sur moyennes GT/Pred) : p-value = 7.5409e-03
  - Biais de Variance (Wilcoxon sur COV GT/Pred)      : p-value = 5.2804e-14
  - Préservation du rang patient (Spearman ρ)         : ρ = 1.000 (p = 8.6670e-112)

📊 RÉSULTATS POUR LA VOI : LESION
🔹 aRE  : 10.85% ± 5.42%
🔹 PSNR : 23.68 ± 5.70 dB
🔹 SSIM : 0.9872 ± 0.0297
🔹 COV  : GT = 0.302 | Pred = 0.273

🧪 Tests Statistiques (N = 62 patients valides)
  - Biais d'Intensité (Wilcoxon sur moyennes GT/Pred) : p-value = 7.5778e-12
  - Biais de Variance (Wilcoxon sur COV GT/Pred)      : p-value = 9.9537e-10
  - Préservation du rang patient (Spearman ρ)         : ρ = 0.989 (p = 3.0623e-51)

📊 RÉSULTATS POUR LA VOI : LIVER
🔹 aRE  : 4.69% ± 0.84%
🔹 PSNR : 27.64 ± 1.09 dB
🔹 SSIM : 0.9371 ± 0.0108
🔹 COV  : GT = 0.110 | Pred = 0.069

🧪 T

In [191]:
resultats = evaluate(
    root_dir='../outputs/pseudo-earl/nantes/',
    sphere_vois=['liver', 'lung', 'brain'],  # Celles-ci seront en sphère de 20mm
    mask_filename='mask.nii.gz',             # Les autres, comme 'spleen', utiliseront le masque entier
    gt_filename='earl2', 
    pred_filename='gaussian-earl2'
)

display_statistics(resultats)


🚀 Lancement de l'évaluation sur :  (88 patients)
🎯 Organes évalués par sphère (20.0mm) : ['liver', 'lung', 'brain']


Calcul des Métriques: 100%|██████████| 88/88 [00:05<00:00, 17.28it/s]


📊 RÉSULTATS POUR LA VOI : BRAIN
🔹 aRE  : 13.17% ± 2.70%
🔹 PSNR : 25.43 ± 0.75 dB
🔹 SSIM : 0.9669 ± 0.0052
🔹 COV  : GT = 0.540 | Pred = 0.443

🧪 Tests Statistiques (N = 88 patients valides)
  - Biais d'Intensité (Wilcoxon sur moyennes GT/Pred) : p-value = 9.6520e-03
  - Biais de Variance (Wilcoxon sur COV GT/Pred)      : p-value = 3.7320e-16
  - Préservation du rang patient (Spearman ρ)         : ρ = 0.999 (p = 1.6851e-119)

📊 RÉSULTATS POUR LA VOI : LESION
🔹 aRE  : 22.45% ± 9.85%
🔹 PSNR : 17.17 ± 5.30 dB
🔹 SSIM : 0.9545 ± 0.0906
🔹 COV  : GT = 0.310 | Pred = 0.235

🧪 Tests Statistiques (N = 69 patients valides)
  - Biais d'Intensité (Wilcoxon sur moyennes GT/Pred) : p-value = 5.2149e-13
  - Biais de Variance (Wilcoxon sur COV GT/Pred)      : p-value = 5.3219e-12
  - Préservation du rang patient (Spearman ρ)         : ρ = 0.926 (p = 4.4108e-30)

📊 RÉSULTATS POUR LA VOI : LIVER
🔹 aRE  : 3.69% ± 0.61%
🔹 PSNR : 29.05 ± 0.92 dB
🔹 SSIM : 0.9397 ± 0.0090
🔹 COV  : GT = 0.083 | Pred = 0.049

🧪 

### CCC - Bland-Altman

In [192]:
rad_results = evaluate_radiomics_from_master(
    master_df[master_df['Domain'] == 'a100'], 
    gt_mod='earl1',
    pred_mod='gaussian-earl1'
)

display_radiomics_statistics(rad_results)


📊 RÉSULTATS POUR LA VOI : BRAIN
🔹 CCC Moyen Global   : 0.783 (p < 0.001)
🔹 Biais Relatif      : -9.03% ± 14.80% (p < 0.001)
   (Testé sur N = 50 paires)

  RÉSULTATS PAR FAMILLE DE RADIOMIQUES :
| Family     | CCC [95% CI]         | Bias (%)   | SD (%)   | 95% LoA (%)       |
|:-----------|:---------------------|:-----------|:---------|:------------------|
| firstorder | 0.906 [0.845, 0.945] | -6.99%     | 19.97%   | [-46.13%, 32.14%] |
| glcm       | 0.720 [0.568, 0.826] | -14.25%    | 5.87%    | [-25.75%, -2.74%] |
| gldm       | 0.802 [0.689, 0.878] | -12.87%    | 13.78%   | [-39.88%, 14.13%] |
| glrlm      | 0.863 [0.772, 0.919] | -3.34%     | 10.81%   | [-24.53%, 17.85%] |
| glszm      | 0.667 [0.493, 0.791] | -3.23%     | 27.91%   | [-57.92%, 51.47%] |
| ngtdm      | 0.699 [0.541, 0.812] | -17.24%    | 12.77%   | [-42.27%, 7.80%]  |

📊 RÉSULTATS POUR LA VOI : LESION
🔹 CCC Moyen Global   : 0.748 (p < 0.001)
🔹 Biais Relatif      : -17.30% ± 42.07% (p < 0.001)
   (Testé sur N = 39 

In [193]:
rad_results = evaluate_radiomics_from_master(
    master_df[master_df['Domain'] == 'a100'], 
    gt_mod='earl2',
    pred_mod='gaussian-earl2'
)

display_radiomics_statistics(rad_results)


📊 RÉSULTATS POUR LA VOI : BRAIN
🔹 CCC Moyen Global   : 0.941 (p < 0.001)
🔹 Biais Relatif      : -0.76% ± 6.60% (p < 0.001)
   (Testé sur N = 50 paires)

  RÉSULTATS PAR FAMILLE DE RADIOMIQUES :
| Family     | CCC [95% CI]         | Bias (%)   | SD (%)   | 95% LoA (%)       |
|:-----------|:---------------------|:-----------|:---------|:------------------|
| firstorder | 0.995 [0.992, 0.997] | -1.33%     | 1.79%    | [-4.83%, 2.18%]   |
| glcm       | 0.948 [0.913, 0.970] | -2.64%     | 2.59%    | [-7.71%, 2.42%]   |
| gldm       | 0.907 [0.853, 0.944] | 0.29%      | 10.35%   | [-20.00%, 20.59%] |
| glrlm      | 0.928 [0.885, 0.956] | 0.73%      | 8.75%    | [-16.43%, 17.89%] |
| glszm      | 0.925 [0.875, 0.956] | 1.49%      | 12.62%   | [-23.25%, 26.23%] |
| ngtdm      | 0.900 [0.838, 0.940] | -4.60%     | 6.51%    | [-17.35%, 8.16%]  |

📊 RÉSULTATS POUR LA VOI : LESION
🔹 CCC Moyen Global   : 0.962 (p < 0.001)
🔹 Biais Relatif      : -4.17% ± 13.96% (p < 0.001)
   (Testé sur N = 39 pa

In [194]:
rad_results = evaluate_radiomics_from_master(
    master_df[master_df['Domain'] == 'chb'], 
    gt_mod='earl1',
    pred_mod='gaussian-earl1'
)

display_radiomics_statistics(rad_results)


📊 RÉSULTATS POUR LA VOI : BRAIN
🔹 CCC Moyen Global   : 0.616 (p < 0.001)
🔹 Biais Relatif      : -7.53% ± 36.34% (p < 0.001)
   (Testé sur N = 99 paires)

  RÉSULTATS PAR FAMILLE DE RADIOMIQUES :
| Family     | CCC [95% CI]         | Bias (%)   | SD (%)   | 95% LoA (%)        |
|:-----------|:---------------------|:-----------|:---------|:-------------------|
| firstorder | 0.873 [0.820, 0.912] | -2.41%     | 51.17%   | [-102.71%, 97.88%] |
| glcm       | 0.495 [0.339, 0.626] | -15.22%    | 48.70%   | [-110.68%, 80.23%] |
| gldm       | 0.634 [0.508, 0.735] | -10.74%    | 25.97%   | [-61.64%, 40.16%]  |
| glrlm      | 0.662 [0.543, 0.757] | -1.07%     | 18.99%   | [-38.29%, 36.15%]  |
| glszm      | 0.468 [0.306, 0.604] | 1.07%      | 31.75%   | [-61.15%, 63.30%]  |
| ngtdm      | 0.547 [0.413, 0.661] | -28.31%    | 22.87%   | [-73.14%, 16.52%]  |

📊 RÉSULTATS POUR LA VOI : LESION
🔹 CCC Moyen Global   : 0.725 (p < 0.001)
🔹 Biais Relatif      : -16.54% ± 37.25% (p < 0.001)
   (Testé sur

In [195]:
rad_results = evaluate_radiomics_from_master(
    master_df[master_df['Domain'] == 'rennes'], 
    gt_mod='earl1',
    pred_mod='gaussian-earl1'
)

display_radiomics_statistics(rad_results)


📊 RÉSULTATS POUR LA VOI : BRAIN
🔹 CCC Moyen Global   : 0.860 (p < 0.001)
🔹 Biais Relatif      : -4.92% ± 9.70% (p < 0.001)
   (Testé sur N = 75 paires)

  RÉSULTATS PAR FAMILLE DE RADIOMIQUES :
| Family     | CCC [95% CI]         | Bias (%)   | SD (%)   | 95% LoA (%)       |
|:-----------|:---------------------|:-----------|:---------|:------------------|
| firstorder | 0.959 [0.937, 0.974] | -2.96%     | 2.78%    | [-8.41%, 2.49%]   |
| glcm       | 0.819 [0.736, 0.878] | -8.37%     | 4.20%    | [-16.59%, -0.15%] |
| gldm       | 0.854 [0.786, 0.903] | -6.60%     | 13.60%   | [-33.26%, 20.07%] |
| glrlm      | 0.896 [0.843, 0.932] | -1.35%     | 10.25%   | [-21.44%, 18.74%] |
| glszm      | 0.818 [0.731, 0.879] | -2.13%     | 21.64%   | [-44.56%, 40.29%] |
| ngtdm      | 0.739 [0.634, 0.821] | -11.12%    | 10.11%   | [-30.93%, 8.69%]  |

📊 RÉSULTATS POUR LA VOI : LESION
🔹 CCC Moyen Global   : 0.877 (p < 0.001)
🔹 Biais Relatif      : -7.39% ± 34.81% (p < 0.001)
   (Testé sur N = 62 pa

In [196]:
rad_results = evaluate_radiomics_from_master(
    master_df[master_df['Domain'] == 'nantes'], 
    gt_mod='earl2',
    pred_mod='gaussian-earl2'
)

display_radiomics_statistics(rad_results)


📊 RÉSULTATS POUR LA VOI : BRAIN
🔹 CCC Moyen Global   : 0.721 (p < 0.001)
🔹 Biais Relatif      : -10.16% ± 12.61% (p < 0.001)
   (Testé sur N = 88 paires)

  RÉSULTATS PAR FAMILLE DE RADIOMIQUES :
| Family     | CCC [95% CI]         | Bias (%)   | SD (%)   | 95% LoA (%)       |
|:-----------|:---------------------|:-----------|:---------|:------------------|
| firstorder | 0.881 [0.827, 0.920] | -6.81%     | 6.92%    | [-20.38%, 6.76%]  |
| glcm       | 0.654 [0.531, 0.751] | -16.18%    | 6.69%    | [-29.29%, -3.07%] |
| gldm       | 0.771 [0.679, 0.840] | -12.20%    | 16.71%   | [-44.96%, 20.55%] |
| glrlm      | 0.834 [0.760, 0.888] | -4.34%     | 11.95%   | [-27.77%, 19.09%] |
| glszm      | 0.544 [0.388, 0.671] | -5.42%     | 24.25%   | [-52.96%, 42.11%] |
| ngtdm      | 0.537 [0.397, 0.657] | -21.31%    | 14.87%   | [-50.46%, 7.84%]  |

📊 RÉSULTATS POUR LA VOI : LESION
🔹 CCC Moyen Global   : 0.735 (p < 0.001)
🔹 Biais Relatif      : -22.97% ± 40.62% (p < 0.001)
   (Testé sur N = 69

### Quantitative evaluation

In [198]:
import os
import numpy as np
import SimpleITK as sitk
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm

# Désactiver les warnings SimpleITK
sitk.ProcessObject.SetGlobalWarningDisplay(False)


def calculate_suv_peak(image_arr, mask_arr, spacing, radius_mm=6.2035):
    """Calcule le SUV Peak (sphère de ~1 mL autour du voxel le plus chaud du masque)."""
    masked_img = np.where(mask_arr > 0, image_arr, -np.inf)
    if np.all(masked_img == -np.inf): 
        return np.nan
        
    # Voxel le plus chaud dans la VOI
    idx_z, idx_y, idx_x = np.unravel_index(np.argmax(masked_img), masked_img.shape)
    sp_x, sp_y, sp_z = spacing
    
    # Bounding box pour la sphère de 1 cm3
    rad_z, rad_y, rad_x = int(np.ceil(radius_mm / sp_z)), int(np.ceil(radius_mm / sp_y)), int(np.ceil(radius_mm / sp_x))
    
    z_min, z_max = max(0, idx_z - rad_z), min(image_arr.shape[0], idx_z + rad_z + 1)
    y_min, y_max = max(0, idx_y - rad_y), min(image_arr.shape[1], idx_y + rad_y + 1)
    x_min, x_max = max(0, idx_x - rad_x), min(image_arr.shape[2], idx_x + rad_x + 1)
    
    box_arr = image_arr[z_min:z_max, y_min:y_max, x_min:x_max]
    zz, yy, xx = np.ogrid[z_min:z_max, y_min:y_max, x_min:x_max]
    
    dist2 = ((zz - idx_z) * sp_z)**2 + ((yy - idx_y) * sp_y)**2 + ((xx - idx_x) * sp_x)**2
    sphere_mask = dist2 <= (radius_mm**2)
    
    return np.mean(box_arr[sphere_mask])

def extract_suv_metrics(gt_arr, pred_arr, mask_arr, spacing):
    """Extrait directement les SUV Mean, Max et Peak."""
    voxels_gt = gt_arr[mask_arr > 0]
    voxels_pred = pred_arr[mask_arr > 0]
    
    if len(voxels_gt) == 0: 
        return None

    return {
        "SUV_Mean_GT": np.mean(voxels_gt), 
        "SUV_Mean_Pred": np.mean(voxels_pred),
        "SUV_Max_GT": np.max(voxels_gt), 
        "SUV_Max_Pred": np.max(voxels_pred),
        "SUV_Peak_GT": calculate_suv_peak(gt_arr, mask_arr, spacing),
        "SUV_Peak_Pred": calculate_suv_peak(pred_arr, mask_arr, spacing)
    }


def process_single_subject(args):
    subject_id, subject_path, gt_filename, pred_filename, mask_filename = args
    results_dict = {}
    
    if not os.path.isdir(subject_path): 
        return subject_id, None
        
    voi_folders = [f for f in os.listdir(subject_path) if os.path.isdir(os.path.join(subject_path, f))]
    
    for voi in voi_folders:
        voi_path = os.path.join(subject_path, voi)
        files = os.listdir(voi_path)
        
        # Sécurisation des extensions
        gt_file = gt_filename if gt_filename.endswith('.nii.gz') else f"{gt_filename}.nii.gz"
        pred_file = pred_filename if pred_filename.endswith('.nii.gz') else f"{pred_filename}.nii.gz"
        
        if gt_file not in files or pred_file not in files or mask_filename not in files: 
            continue
            
        try:
            # Chargement des images
            gt_img = sitk.ReadImage(os.path.join(voi_path, gt_file))
            pred_img = sitk.ReadImage(os.path.join(voi_path, pred_file))
            mask_img = sitk.ReadImage(os.path.join(voi_path, mask_filename))
                    
            gt_arr = sitk.GetArrayFromImage(gt_img)
            pred_arr = sitk.GetArrayFromImage(pred_img)
            mask_arr = sitk.GetArrayFromImage(mask_img)
            spacing = gt_img.GetSpacing()
            
            # Extraction
            metrics = extract_suv_metrics(gt_arr, pred_arr, mask_arr, spacing)
            if metrics: 
                results_dict[voi] = metrics
        except Exception:
            pass 
            
    return subject_id, results_dict

def evaluate_suv_pipeline(root_dir, include_only=None, gt_filename="EARL", pred_filename="predicted", mask_filename="mask.nii.gz", num_workers=None):
    if not os.path.exists(root_dir): 
        return {}

    subjects = [s for s in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, s))]
    if include_only: 
        subjects = [s for s in subjects if s in include_only]
    
    tasks = [(subj, os.path.join(root_dir, subj), gt_filename, pred_filename, mask_filename) for subj in subjects]
    center_results = {}
    
    print(f"🚀 Lancement de l'évaluation SUV sur : {os.path.basename(root_dir)} ({len(subjects)} patients)")
    
    with ProcessPoolExecutor(max_workers=num_workers) as executor:
        futures = {executor.submit(process_single_subject, task): task for task in tasks}
        for future in tqdm(as_completed(futures), total=len(tasks), desc="Extraction NIfTI"):
            subject_id, subj_results = future.result()
            if subj_results: 
                center_results[subject_id] = subj_results

    return center_results


def print_quantitative_stats(metric_name, gt_list, pred_list):
    gt, pred = np.array(gt_list), np.array(pred_list)
    valid_mask = ~(np.isnan(gt) | np.isnan(pred))
    gt, pred = gt[valid_mask], pred[valid_mask]
    
    if len(gt) == 0:
        print(f"🔹 {metric_name.ljust(10)}: Pas de données valides")
        return

    # Brut
    delta = pred - gt
    bias_raw = np.mean(delta)
    std_raw = np.std(delta, ddof=1) if len(delta) > 1 else 0
    loa_inf_raw, loa_sup_raw = bias_raw - 1.96 * std_raw, bias_raw + 1.96 * std_raw
    
    # Pourcentage (%)
    safe_gt = np.where(gt == 0, 1e-8, gt)
    delta_pct = (delta / safe_gt) * 100.0
    bias_pct = np.mean(delta_pct)
    std_pct = np.std(delta_pct, ddof=1) if len(delta_pct) > 1 else 0
    loa_inf_pct, loa_sup_pct = bias_pct - 1.96 * std_pct, bias_pct + 1.96 * std_pct
    
    print(f"🔹 {metric_name.ljust(10)}:")
    print(f"   ↳ Valeurs Brutes : Delta = {bias_raw:+.3f} | SD = {std_raw:.3f} | 95% LoA = [{loa_inf_raw:+.3f}, {loa_sup_raw:+.3f}]")
    print(f"   ↳ Relatif (%)    : Delta = {bias_pct:+.2f}% | SD = {std_pct:.2f}% | 95% LoA = [{loa_inf_pct:+.2f}%, {loa_sup_pct:+.2f}%]")

def display_suv_statistics(res):
    if not res:
        print("⚠️ Aucun résultat à afficher.")
        return
        
    vois_trouvees = set()
    for subj in res.values(): 
        vois_trouvees.update(subj.keys())
        
    for voi in sorted(vois_trouvees):
        print("\n" + "="*80)
        print(f"📊 ANALYSE QUANTITATIVE : {voi.upper()}")
        print("="*80)
        
        mean_gt, mean_pred, max_gt, max_pred, peak_gt, peak_pred = [], [], [], [], [], []
        
        for subj_data in res.values():
            if voi in subj_data:
                m = subj_data[voi]
                mean_gt.append(m["SUV_Mean_GT"])
                mean_pred.append(m["SUV_Mean_Pred"])
                max_gt.append(m["SUV_Max_GT"])
                max_pred.append(m["SUV_Max_Pred"])
                peak_gt.append(m["SUV_Peak_GT"])
                peak_pred.append(m["SUV_Peak_Pred"])
        
        print(f"Nombre de patients valides : {len(mean_gt)}\n")
        print_quantitative_stats("SUV Mean", mean_gt, mean_pred)
        print_quantitative_stats("SUV Max", max_gt, max_pred)
        print_quantitative_stats("SUV Peak", peak_gt, peak_pred)


In [200]:
suv_results = evaluate_suv_pipeline(
    root_dir='../outputs/pseudo-earl/a100/', 
    gt_filename='earl1', 
    pred_filename='pseudo-earl1', 
    num_workers=8
)

display_suv_statistics(suv_results)

🚀 Lancement de l'évaluation SUV sur :  (50 patients)


Extraction NIfTI: 100%|██████████| 50/50 [00:01<00:00, 25.53it/s]


📊 ANALYSE QUANTITATIVE : BRAIN
Nombre de patients valides : 50

🔹 SUV Mean  :
   ↳ Valeurs Brutes : Delta = -0.001 | SD = 0.003 | 95% LoA = [-0.006, +0.003]
   ↳ Relatif (%)    : Delta = -0.02% | SD = 0.04% | 95% LoA = [-0.10%, +0.05%]
🔹 SUV Max   :
   ↳ Valeurs Brutes : Delta = +0.022 | SD = 0.061 | 95% LoA = [-0.096, +0.141]
   ↳ Relatif (%)    : Delta = +0.20% | SD = 0.50% | 95% LoA = [-0.77%, +1.18%]
🔹 SUV Peak  :
   ↳ Valeurs Brutes : Delta = -0.043 | SD = 0.124 | 95% LoA = [-0.285, +0.200]
   ↳ Relatif (%)    : Delta = -0.39% | SD = 1.24% | 95% LoA = [-2.82%, +2.03%]

📊 ANALYSE QUANTITATIVE : LESION
Nombre de patients valides : 39

🔹 SUV Mean  :
   ↳ Valeurs Brutes : Delta = +0.004 | SD = 0.042 | 95% LoA = [-0.078, +0.085]
   ↳ Relatif (%)    : Delta = +0.11% | SD = 0.76% | 95% LoA = [-1.38%, +1.60%]
🔹 SUV Max   :
   ↳ Valeurs Brutes : Delta = +0.053 | SD = 0.191 | 95% LoA = [-0.321, +0.427]
   ↳ Relatif (%)    : Delta = +0.43% | SD = 1.50% | 95% LoA = [-2.51%, +3.36%]
🔹 SUV Pea

In [201]:
suv_results = evaluate_suv_pipeline(
    root_dir='../outputs/pseudo-earl/a100/', 
    gt_filename='earl2', 
    pred_filename='pseudo-earl2', 
    num_workers=8
)

display_suv_statistics(suv_results)

🚀 Lancement de l'évaluation SUV sur :  (50 patients)


Extraction NIfTI: 100%|██████████| 50/50 [00:02<00:00, 24.76it/s]


📊 ANALYSE QUANTITATIVE : BRAIN
Nombre de patients valides : 50

🔹 SUV Mean  :
   ↳ Valeurs Brutes : Delta = -0.002 | SD = 0.003 | 95% LoA = [-0.008, +0.004]
   ↳ Relatif (%)    : Delta = -0.03% | SD = 0.05% | 95% LoA = [-0.12%, +0.06%]
🔹 SUV Max   :
   ↳ Valeurs Brutes : Delta = -0.006 | SD = 0.125 | 95% LoA = [-0.252, +0.240]
   ↳ Relatif (%)    : Delta = -0.07% | SD = 1.02% | 95% LoA = [-2.07%, +1.92%]
🔹 SUV Peak  :
   ↳ Valeurs Brutes : Delta = +0.004 | SD = 0.111 | 95% LoA = [-0.213, +0.222]
   ↳ Relatif (%)    : Delta = +0.05% | SD = 1.12% | 95% LoA = [-2.15%, +2.25%]

📊 ANALYSE QUANTITATIVE : LESION
Nombre de patients valides : 39

🔹 SUV Mean  :
   ↳ Valeurs Brutes : Delta = +0.008 | SD = 0.041 | 95% LoA = [-0.073, +0.088]
   ↳ Relatif (%)    : Delta = +0.17% | SD = 0.88% | 95% LoA = [-1.55%, +1.89%]
🔹 SUV Max   :
   ↳ Valeurs Brutes : Delta = +0.018 | SD = 0.314 | 95% LoA = [-0.598, +0.634]
   ↳ Relatif (%)    : Delta = +0.29% | SD = 2.23% | 95% LoA = [-4.07%, +4.65%]
🔹 SUV Pea

In [202]:
suv_results = evaluate_suv_pipeline(
    root_dir='../outputs/pseudo-earl/chb/', 
    gt_filename='earl1', 
    pred_filename='pseudo-earl1', 
    num_workers=8
)

display_suv_statistics(suv_results)

🚀 Lancement de l'évaluation SUV sur :  (100 patients)


Extraction NIfTI: 100%|██████████| 100/100 [00:03<00:00, 29.33it/s]


📊 ANALYSE QUANTITATIVE : BRAIN
Nombre de patients valides : 100

🔹 SUV Mean  :
   ↳ Valeurs Brutes : Delta = -0.003 | SD = 0.033 | 95% LoA = [-0.068, +0.061]
   ↳ Relatif (%)    : Delta = -0.04% | SD = 0.45% | 95% LoA = [-0.93%, +0.84%]
🔹 SUV Max   :
   ↳ Valeurs Brutes : Delta = -0.030 | SD = 0.621 | 95% LoA = [-1.246, +1.187]
   ↳ Relatif (%)    : Delta = -0.03% | SD = 3.23% | 95% LoA = [-6.35%, +6.30%]
🔹 SUV Peak  :
   ↳ Valeurs Brutes : Delta = +0.140 | SD = 0.434 | 95% LoA = [-0.710, +0.991]
   ↳ Relatif (%)    : Delta = +1.19% | SD = 3.03% | 95% LoA = [-4.74%, +7.12%]

📊 ANALYSE QUANTITATIVE : LESION
Nombre de patients valides : 82

🔹 SUV Mean  :
   ↳ Valeurs Brutes : Delta = -0.001 | SD = 0.013 | 95% LoA = [-0.026, +0.025]
   ↳ Relatif (%)    : Delta = +0.00% | SD = 0.32% | 95% LoA = [-0.62%, +0.63%]
🔹 SUV Max   :
   ↳ Valeurs Brutes : Delta = +0.003 | SD = 0.034 | 95% LoA = [-0.064, +0.071]
   ↳ Relatif (%)    : Delta = +0.04% | SD = 0.42% | 95% LoA = [-0.78%, +0.86%]
🔹 SUV Pe

In [203]:
suv_results = evaluate_suv_pipeline(
    root_dir='../outputs/pseudo-earl/rennes/', 
    gt_filename='earl1', 
    pred_filename='pseudo-earl1', 
    num_workers=8
)

display_suv_statistics(suv_results)

🚀 Lancement de l'évaluation SUV sur :  (75 patients)


Extraction NIfTI: 100%|██████████| 75/75 [00:02<00:00, 28.45it/s]


📊 ANALYSE QUANTITATIVE : BRAIN
Nombre de patients valides : 65

🔹 SUV Mean  :
   ↳ Valeurs Brutes : Delta = -0.006 | SD = 0.008 | 95% LoA = [-0.022, +0.010]
   ↳ Relatif (%)    : Delta = -0.08% | SD = 0.12% | 95% LoA = [-0.31%, +0.15%]
🔹 SUV Max   :
   ↳ Valeurs Brutes : Delta = -0.029 | SD = 0.139 | 95% LoA = [-0.302, +0.243]
   ↳ Relatif (%)    : Delta = -0.23% | SD = 1.05% | 95% LoA = [-2.29%, +1.84%]
🔹 SUV Peak  :
   ↳ Valeurs Brutes : Delta = +0.006 | SD = 0.263 | 95% LoA = [-0.510, +0.522]
   ↳ Relatif (%)    : Delta = +0.07% | SD = 2.24% | 95% LoA = [-4.31%, +4.45%]

📊 ANALYSE QUANTITATIVE : LESION
Nombre de patients valides : 53

🔹 SUV Mean  :
   ↳ Valeurs Brutes : Delta = -0.003 | SD = 0.086 | 95% LoA = [-0.171, +0.166]
   ↳ Relatif (%)    : Delta = +0.23% | SD = 1.87% | 95% LoA = [-3.44%, +3.89%]
🔹 SUV Max   :
   ↳ Valeurs Brutes : Delta = +0.280 | SD = 1.111 | 95% LoA = [-1.897, +2.457]
   ↳ Relatif (%)    : Delta = +1.01% | SD = 4.45% | 95% LoA = [-7.72%, +9.74%]
🔹 SUV Pea

In [204]:
suv_results = evaluate_suv_pipeline(
    root_dir='../outputs/pseudo-earl/nantes/', 
    gt_filename='earl2', 
    pred_filename='pseudo-earl2', 
    num_workers=8
)

display_suv_statistics(suv_results)

🚀 Lancement de l'évaluation SUV sur :  (88 patients)


Extraction NIfTI: 100%|██████████| 88/88 [00:02<00:00, 30.05it/s]



📊 ANALYSE QUANTITATIVE : BRAIN
Nombre de patients valides : 88

🔹 SUV Mean  :
   ↳ Valeurs Brutes : Delta = -0.001 | SD = 0.001 | 95% LoA = [-0.003, +0.002]
   ↳ Relatif (%)    : Delta = -0.01% | SD = 0.02% | 95% LoA = [-0.05%, +0.02%]
🔹 SUV Max   :
   ↳ Valeurs Brutes : Delta = +0.022 | SD = 0.058 | 95% LoA = [-0.092, +0.137]
   ↳ Relatif (%)    : Delta = +0.18% | SD = 0.47% | 95% LoA = [-0.73%, +1.09%]
🔹 SUV Peak  :
   ↳ Valeurs Brutes : Delta = -0.011 | SD = 0.170 | 95% LoA = [-0.344, +0.321]
   ↳ Relatif (%)    : Delta = -0.12% | SD = 1.50% | 95% LoA = [-3.06%, +2.82%]

📊 ANALYSE QUANTITATIVE : LESION
Nombre de patients valides : 69

🔹 SUV Mean  :
   ↳ Valeurs Brutes : Delta = -0.003 | SD = 0.029 | 95% LoA = [-0.059, +0.054]
   ↳ Relatif (%)    : Delta = -0.02% | SD = 0.40% | 95% LoA = [-0.81%, +0.76%]
🔹 SUV Max   :
   ↳ Valeurs Brutes : Delta = +0.021 | SD = 0.137 | 95% LoA = [-0.247, +0.289]
   ↳ Relatif (%)    : Delta = +0.21% | SD = 0.75% | 95% LoA = [-1.27%, +1.68%]
🔹 SUV Pea